# CP1 · U18 · Încărcare date, alinierea ferestrelor și data card
Surse: **S1** IMU (PAMAP2) · **S2** puls (PAMAP2) · **S3** valori MET (2024 Adult Compendium of Physical Activities).
Rulează de sus în jos; ultima celulă scrie `datacard_U18.md`.

In [1]:
import os, io, glob, zipfile, urllib.request
import numpy as np, pandas as pd

ROOTS = ["/kaggle/input", "/kaggle/working", "."]

def find_subject_files():
    out = []
    for r in ROOTS:
        out += glob.glob(f"{r}/**/subject10?.dat", recursive=True)
    return sorted({f for f in out if "Optional" not in f})

files = find_subject_files()
if not files:
    print("No inputs found, will download...")
    url = "https://archive.ics.uci.edu/static/public/231/pamap2+physical+activity+monitoring.zip"
    urllib.request.urlretrieve(url, "/kaggle/working/pamap2.zip")
    with zipfile.ZipFile("/kaggle/working/pamap2.zip") as z:
        z.extractall("/kaggle/working/pamap2")
    for inner in glob.glob("/kaggle/working/pamap2/**/*.zip", recursive=True):
        with zipfile.ZipFile(inner) as z:
            z.extractall("/kaggle/working/pamap2")
    files = find_subject_files()

print(len(files), "fișiere:", [os.path.basename(f) for f in files])
assert files, "No subject10x.dat file found"

1 fișiere: ['subject101.dat']


In [2]:
# 2) S1 (IMU) și S2 (puls) din PAMAP2
COLS = ["timestamp", "activityID", "heart_rate"]
for loc in ["hand", "chest", "ankle"]:
    COLS += [f"{loc}_temp"]
    COLS += [f"{loc}_acc16_{a}" for a in "xyz"]
    COLS += [f"{loc}_acc6_{a}" for a in "xyz"]
    COLS += [f"{loc}_gyro_{a}" for a in "xyz"]
    COLS += [f"{loc}_mag_{a}" for a in "xyz"]
    COLS += [f"{loc}_orient_{i}" for i in range(4)]

def load_pamap(files):
    parts = []
    for f in files:
        d = pd.read_csv(f, sep=r"\s+", header=None, names=COLS)
        assert d.shape[1] == 54, f"{f}: {d.shape[1]} coloane"
        d.insert(0, "subject", int(os.path.basename(f)[7:10]))
        parts.append(d)
    return pd.concat(parts, ignore_index=True)

raw = load_pamap(files)
S1 = raw.drop(columns=[c for c in raw.columns if "orient" in c or c == "heart_rate"])
S2 = raw.loc[raw["heart_rate"].notna(), ["subject", "timestamp", "activityID", "heart_rate"]].reset_index(drop=True)

print("S1 IMU :", S1.shape)
print("S2 puls:", S2.shape)

S1 IMU : (376417, 42)
S2 puls: (34389, 4)


In [3]:
import re

def find_compendium():
    for r in ROOTS:
        for pat in ("**/*ompendium*.pdf", "**/*ompendium*.txt"):
            hits = glob.glob(f"{r}/{pat}", recursive=True)
            if hits:
                return hits[0]
    return None

def read_text(path):
    if path.lower().endswith(".pdf"):
        try:
            from pypdf import PdfReader
        except ImportError:
            from PyPDF2 import PdfReader
        return "\n".join((p.extract_text() or "") for p in PdfReader(path).pages)
    return open(path, encoding="utf-8").read()

comp_path = find_compendium()
assert comp_path, "Input 'compendium'"
text = read_text(comp_path)

COMP = pd.DataFrame(re.findall(r"(?<!\d)(\d{5})\s+(\d{1,2}\.\d{1,2})\s+([^\n]*)", text),
                    columns=["code", "met", "description"])
COMP["met"] = COMP["met"].astype(float)
COMP = COMP.drop_duplicates("code").set_index("code")
print(os.path.basename(comp_path), "->", len(COMP), "activități în Compendium")
assert len(COMP) > 500, "Parsarea a găsit prea puține rânduri; verifică fișierul"

ACT = {1: "lying", 2: "sitting", 3: "standing", 4: "walking", 5: "running", 6: "cycling",
       7: "Nordic walking", 12: "ascending stairs", 13: "descending stairs",
       16: "vacuum cleaning", 17: "ironing", 24: "rope jumping"}

MET_PICKS = {
    1: "07011",   # Lying quietly, doing nothing, lying in bed awake
    2: "07021",   # Sitting quietly, general
    3: "07040",   # Standing quietly (standing in a line)
    4: "17190",   # Walking, 2.8 to 3.4 mph, level, moderate pace, firm surface
    5: "12020",   # Jogging, general, self-selected pace
    6: "01016",   # Bicycling, self-selected moderate pace
    7: "17302",   # Walking, for exercise, 2.5 to 3.5 mph, with ski poles, Nordic walking, level, moderate pace
    12: "17131",  # Stair climbing, general
    13: "17070",  # Descending stairs
    16: "05043",  # Vacuuming, general, moderate effort
    17: "05070",  # Ironing
    24: "02068",  # Rope skipping exercise, general
}

S3 = pd.DataFrame({
    "activityID": list(ACT),
    "activity": list(ACT.values()),
    "compendium_code": [MET_PICKS[a] for a in ACT],
})
S3["met"] = S3["compendium_code"].map(COMP["met"])
S3["description"] = S3["compendium_code"].map(COMP["description"])
assert S3["met"].notna().all(), f"Coduri negăsite: {S3.loc[S3.met.isna(), 'compendium_code'].tolist()}"
S3

1_2024-adult-compendium_1_2024.pdf -> 1111 activități în Compendium


,activityID,activity,compendium_code,met,description
0,1,lying,07011,1.0,"Lying quietly, doing nothing, lying in bed awa..."
1,2,sitting,07021,1.0,"Sitting quietly, general"
2,3,standing,07040,1.3,Standing quietly (standing in a line)
3,4,walking,17190,3.8,"Walking, 2.8 to 3.4 mph, level, moderate pace,..."
4,5,running,12020,7.5,"Jogging, general, self-selected pace"
5,6,cycling,01016,7.0,"Bicycling, self-selected moderate pace"
6,7,Nordic walking,17302,4.3,"Walking, for exercise, 2.5 to 3.5 mph (4.0 to ..."
7,12,ascending stairs,17131,6.8,"Stair climbing, general"
8,13,descending stairs,17070,3.5,Descending stairs
9,16,vacuum cleaning,05043,3.0,"Vacuuming, general, moderate effort"


In [4]:
# 4) Alinierea: persoană × fereastră de 5 s + raport de acoperire
WIN = 5
LOCS = ["hand", "chest", "ankle"]

imu = S1.copy()
for l in LOCS:
    imu[f"{l}_acc_mag"] = np.sqrt(imu[f"{l}_acc16_x"]**2 + imu[f"{l}_acc16_y"]**2 + imu[f"{l}_acc16_z"]**2)
imu["win"] = (imu["timestamp"] // WIN).astype(int)

def dominant(s):
    return s.mode().iat[0]

agg = {"n_imu": ("timestamp", "size"), "activityID": ("activityID", dominant),
       "purity": ("activityID", lambda s: (s == s.mode().iat[0]).mean())}
for l in LOCS:
    agg[f"{l}_acc_mean"] = (f"{l}_acc_mag", "mean")
    agg[f"{l}_acc_std"] = (f"{l}_acc_mag", "std")
A = imu.groupby(["subject", "win"]).agg(**agg).reset_index()

S2w = S2.assign(win=(S2["timestamp"] // WIN).astype(int)) \
        .groupby(["subject", "win"]).agg(hr_mean=("heart_rate", "mean"), n_hr=("heart_rate", "size")).reset_index()

aligned = A.merge(S2w, on=["subject", "win"], how="left") \
           .merge(S3[["activityID", "met"]], on="activityID", how="left")
aligned["n_hr"] = aligned["n_hr"].fillna(0).astype(int)

# păstrăm doar ferestre cu activitate definită (0 = tranziție) și dominantă
aligned = aligned[(aligned["activityID"] != 0) & (aligned["purity"] >= 0.8)].reset_index(drop=True)

def pct(s): return round(100 * s.mean(), 1)
cov = pd.DataFrame({
    "ferestre": aligned.groupby("subject").size(),
    "cu_puls_%": aligned.groupby("subject")["hr_mean"].apply(lambda s: pct(s.notna())),
    "cu_MET_%": aligned.groupby("subject")["met"].apply(lambda s: pct(s.notna())),
    "toate_3_surse_%": aligned.groupby("subject").apply(lambda d: pct(d["hr_mean"].notna() & d["met"].notna())),
})
cov.loc["TOTAL"] = [len(aligned), pct(aligned["hr_mean"].notna()), pct(aligned["met"].notna()),
                    pct(aligned["hr_mean"].notna() & aligned["met"].notna())]
print("Tabelul aliniat:", aligned.shape)
display(cov)           
aligned.head(10)

Tabelul aliniat: (492, 14)


/tmp/ipykernel_48/209270710.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  "toate_3_surse_%": aligned.groupby("subject").apply(lambda d: pct(d["hr_mean"].notna() & d["met"].notna())),


,ferestre,cu_puls_%,cu_MET_%,toate_3_surse_%
subject,,,,
101,492.0,100.0,100.0,100.0
TOTAL,492.0,100.0,100.0,100.0


,subject,win,n_imu,activityID,purity,hand_acc_mean,hand_acc_std,chest_acc_mean,chest_acc_std,ankle_acc_mean,ankle_acc_std,hr_mean,n_hr,met
0,101,8,500,1,1.0,9.862423,1.005769,9.852348,0.150709,9.848242,0.080721,102.000000,46,1.0
1,101,9,500,1,1.0,10.014998,1.338387,9.914067,0.962700,10.257678,1.444791,102.739130,46,1.0
2,101,10,500,1,1.0,9.800909,0.279656,9.658485,0.335686,10.136656,1.324937,105.434783,46,1.0
3,101,11,500,1,1.0,9.815870,0.367383,9.658269,0.165179,10.102450,0.089372,106.266667,45,1.0
4,101,12,500,1,1.0,9.790589,0.092661,9.648196,0.134683,10.112903,0.082980,103.565217,46,1.0
5,101,13,500,1,1.0,9.794995,0.086442,9.648522,0.131074,10.109229,0.083430,99.913043,46,1.0
6,101,14,500,1,1.0,9.794744,0.088000,9.647994,0.127938,10.116282,0.082907,96.511111,45,1.0
7,101,15,500,1,1.0,9.801407,0.093397,9.649106,0.128430,10.116173,0.083292,94.326087,46,1.0
8,101,16,500,1,1.0,9.797655,0.090165,9.651045,0.127589,10.117315,0.085893,93.195652,46,1.0
9,101,17,500,1,1.0,9.798639,0.089384,9.641445,0.125295,10.120902,0.084023,92.586957,46,1.0


In [5]:
dur = raw.groupby("subject")["timestamp"].agg(lambda s: s.max() - s.min()).sum()
dt_imu = np.median(np.diff(raw.loc[raw.subject == raw.subject.iloc[0], "timestamp"].values))
hr_rate = len(S2) / dur
nan_hr = 100 * raw["heart_rate"].isna().mean()
win_hr = cov.loc["TOTAL", "cu_puls_%"]
max_gap = S2.groupby("subject")["timestamp"].apply(lambda t: t.diff().max()).max()
bad_hr = 100 * ((S2["heart_rate"] < 30) | (S2["heart_rate"] > 220)).mean()
nan_imu = 100 * S1["hand_acc16_x"].isna().mean()
acts = ", ".join(f"{k} {v}" for k, v in ACT.items())
met_ok = int(S3["met"].notna().sum())

card = f"""# Data card · U18 (persoană × fereastră de {WIN} s)

## S1 · IMU (accelerometru + giroscop)
- **Proveniență:** PAMAP2 Physical Activity Monitoring, UCI Machine Learning Repository (Reiss & Stricker, 2012), https://archive.ics.uci.edu/dataset/231
- **Perioadă:** {len(files)} subiecți, {dur/3600:.1f} h în total; timp relativ în secunde de la începutul înregistrării (setul nu conține date calendaristice).
- **Rezoluție spațială:** 3 poziții pe corp (mână, piept, gleznă); fără componentă geografică.
- **Rezoluție temporală:** ~{1/dt_imu:.0f} Hz. După aliniere: ferestre de {WIN} s.
- **Unități:** accelerație m/s² (±16g și ±6g), giroscop rad/s, magnetometru µT, temperatură °C.
- **Goluri:** {nan_imu:.1f}% valori lipsă la accelerometrul de la mână; activitatea 0 (tranziții) exclusă din analiză; orientarea e invalidă și nu se folosește.
- **Licență:** CC BY 4.0.
- **n × d:** {len(S1):,} rânduri × {S1.shape[1]} coloane. Activități folosite: {acts}.

## S2 · Puls (monitor separat)
- **Proveniență:** același set PAMAP2, dar măsurat de un monitor de puls separat de senzorii IMU.
- **Perioadă:** aceeași ca la S1.
- **Rezoluție spațială:** nu se aplică (un singur punct de măsură, la persoană).
- **Rezoluție temporală:** ~{hr_rate:.1f} Hz efectiv ({len(S2):,} valori de puls). După aliniere: media pe fereastră de {WIN} s.
- **Unități:** bătăi pe minut (bpm).
- **Goluri:** {nan_hr:.1f}% din rândurile fișierelor nu au puls, dar asta se datorează frecvenței (~9 Hz față de 100 Hz), nu unor date lipsă. La nivel de fereastră de {WIN} s, {win_hr}% dintre ferestre au cel puțin o valoare de puls; cel mai lung interval fără puls este {max_gap:.1f} s; {bad_hr:.2f}% dintre valori sunt în afara intervalului 30–220 bpm.
- **Licență:** CC BY 4.0.
- **n × d:** {len(S2):,} rânduri × 1 variabilă de puls.

## S3 · Valori MET pe activitate (referință de efort)
- **Proveniență:** 2024 Adult Compendium of Physical Activities (Herrmann et al., 2024), https://pacompendium.com; fiecare activitate din PAMAP2 e legată de un rând al Compendium (cod în `S3`).
- **Perioadă:** tabel de referință, fără dimensiune temporală (versiunea 2024).
- **Rezoluție spațială / temporală:** nu se aplică; o valoare pe tip de activitate.
- **Unități:** MET (multipli ai consumului de energie în repaus).
- **Goluri:** {met_ok} din {len(ACT)} activități au valoare MET asociată; alegerea rândului Compendium este o decizie de aliniere semantică (PAMAP2 nu are viteze).
- **Licență:** de verificat pe pacompendium.com înainte de predare.
- **n × d:** {len(S3)} rânduri (activități) × 3 coloane utile (cod, MET, descriere).

## Aliniere
{len(aligned):,} ferestre de {WIN} s (persoană × fereastră), păstrate doar cele cu activitate dominantă (≥ 80% din rânduri). Acoperire: puls {cov.loc['TOTAL','cu_puls_%']}%, MET {cov.loc['TOTAL','cu_MET_%']}%, toate trei {cov.loc['TOTAL','toate_3_surse_%']}%.
"""
with open("datacard_U18.md", "w", encoding="utf-8") as f:
    f.write(card)
print(card)

# Data card · U18 (persoană × fereastră de 5 s)

## S1 · IMU (accelerometru + giroscop)
- **Proveniență:** PAMAP2 Physical Activity Monitoring, UCI Machine Learning Repository (Reiss & Stricker, 2012), https://archive.ics.uci.edu/dataset/231
- **Perioadă:** 1 subiecți, 1.0 h în total; timp relativ în secunde de la începutul înregistrării (setul nu conține date calendaristice).
- **Rezoluție spațială:** 3 poziții pe corp (mână, piept, gleznă); fără componentă geografică.
- **Rezoluție temporală:** ~100 Hz. După aliniere: ferestre de 5 s.
- **Unități:** accelerație m/s² (±16g și ±6g), giroscop rad/s, magnetometru µT, temperatură °C.
- **Goluri:** 0.4% valori lipsă la accelerometrul de la mână; activitatea 0 (tranziții) exclusă din analiză; orientarea e invalidă și nu se folosește.
- **Licență:** CC BY 4.0.
- **n × d:** 376,417 rânduri × 42 coloane. Activități folosite: 1 lying, 2 sitting, 3 standing, 4 walking, 5 running, 6 cycling, 7 Nordic walking, 12 ascending stairs, 13 descending st